# BM25 и dense через RRF с cross-encoder реранкером

Эксперимент сравнивает четыре варианта на демонстрационной calibration-выборке. Holdout не используется. Реранкер не гарантирует улучшения и не может восстановить документ, отсутствующий среди кандидатов. Качество и latency рассматриваются вместе. Подробное объяснение добавлено в Word.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
REPORT = ROOT / 'reports' / 'hybrid_comparison.json'


## Повторный эксперимент

Для настоящего нейросетевого прогона установите `requirements-hybrid.txt` в отдельное окружение и выберите его ядром PyCharm. Модели и веса сохраняются в `work/hf_cache/`. Загрузка занимает время и место. Для просмотра уже измеренных результатов torch не требуется. Чтобы повторить прогон, задайте RUN_EXPERIMENT=True. Не используйте прежний demo holdout для выбора нового метода.


In [ ]:
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    from scripts.compare_hybrid import run
    result, run_dir = run(ROOT / 'data/demo', ROOT / 'work/hybrid_runs')
    print('Артефакты:', run_dir)
else:
    print('Показываем сохранённый отчёт; новый расчёт не выполняется.')


In [ ]:
report = json.loads(REPORT.read_text(encoding='utf-8'))
table = pd.DataFrame([{key: value for key, value in metrics.items() if key != 'languages'} | {'method': method} for method, metrics in report['metrics'].items()])
display(table)
print('Выбор только по calibration:', report['selected_on_calibration'])
print('Holdout открыт:', report['holdout_opened'])
print('Зафиксированные модели:', report['revisions'])


## Языки и изменения выдачи

Сравните top1_accuracy и MRR@5 до и после reranking. Положительный top1_delta означает исправление top-1, отрицательный — регрессию. Ни BM25 score, ни cosine, ни RRF, ни logit реранкера не являются готовой вероятностью правильного ответа. Для отказов потребуется отдельная calibration каждого варианта.


In [ ]:
language_rows = []
for method, metrics in report['metrics'].items():
    for language, values in metrics['languages'].items():
        language_rows.append({'method': method, 'language': language, **values})
display(pd.DataFrame(language_rows))
display(pd.DataFrame(report['reranker_changes']))


## Формула RRF

RRF(d) = сумма по выдачам 1 / (c + rank(d)). Ранги начинаются с 1; отсутствующий кандидат даёт нулевой вклад. В этом эксперименте c=60. Каждая ветка возвращает 4 чанка, их объединение даёт до 8 кандидатов; реранкер оценивает первые 6, финальная выдача содержит до 5. Объединение выполняется по chunk_id, а метрики — по уникальным document_id. RRF — один из способов fusion; после надёжной нормализации возможна и взвешенная сумма score.
